# RI-JK UHF Hessian：CP-HF 分解 (1) 骨架 Fock 一阶导数

本文档对应 `02-4-decomp_cphf_1.ipynb` 的 UHF 版本。重点在于：

1. 骨架 Fock 一阶导数 `f1ao` 的分解：`h1ao`（与自旋无关）+ `j1ao`（依赖总密度，与自旋无关）− `k1ao`（按自旋分通道）。
2. 注意 UHF 中 K 不再带 RHF 中的 `0.5` 因子：`f1ao_σ = h1ao + j1ao - k1ao_σ`。
3. 维度说明：UHF 下 `mo1` / `mo_e1` 的自旋通道维度不规则（`nmo×nocca` 与 `nmo×noccb` 不同），因此**无法**用 5-dim 张量表示，需要用列表/元组形式存储。

记号约定（参见 `03-2-skeleton_deriv.md` 第 1.0 节）：自旋通道使用 `x`（einsum）/ `σ`（公式），基函数仍用 `u,v,k,l`，分子轨道用 `i,j` / `p,q`。

In [1]:
from pyscf import gto, scf, lib, df, hessian
from pyscf.df.hessian import uhf as df_uhf_hess
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_u_hf_decomp.npz")["de_cphf"]

In [6]:
α, β = 0, 1

mo_coeff = mf.mo_coeff           # shape (2, nao, nmo)
mo_occ = mf.mo_occ               # shape (2, nmo)
mo_energy = mf.mo_energy         # shape (2, nmo)
nao = mo_coeff.shape[1]
nmo = mo_coeff.shape[2]

# 列表形式存储按自旋分通道的占据轨道、虚轨道、能量等
mocc = [mo_coeff[x][:, mo_occ[x] > 0] for x in (α, β)]
mvir = [mo_coeff[x][:, mo_occ[x] == 0] for x in (α, β)]
mocc_2 = [mocc[α], mocc[β]]  # UHF 下 occ=1，无需 sqrt 缩放
nocc = [mocc[x].shape[1] for x in (α, β)]
nvir = [mvir[x].shape[1] for x in (α, β)]
eocc = [mo_energy[x][mo_occ[x] > 0] for x in (α, β)]
evir = [mo_energy[x][mo_occ[x] == 0] for x in (α, β)]

# 密度矩阵：按自旋分量存储，并提供总密度形式
dm0 = np.zeros((2, nao, nao))
dm0[α] = mocc[α] @ mocc[α].T
dm0[β] = mocc[β] @ mocc[β].T
dm0_s = dm0.sum(axis=0)

natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao

## Overview

这里先用 `mf_hess.make_h1` / `mf_hess.solve_mo1` 跑一遍 PySCF 自带的 CP-HF 流程，得到参考 `f1ao`、`mo1_bra`、`mo_e1`，作为后续手动分解的核验基准。

注意：
- `f1ao` 是 tuple `(f1ao_a, f1ao_b)`，每一项 shape `[natm, 3, nao, nao]`。
- `mo1_bra` 是 tuple `(mo1_bra_a, mo1_bra_b)`，分别 shape `[natm, 3, nao, nocc_α]` 与 `[natm, 3, nao, nocc_β]`，**两者无法合并为单个张量**。
- `mo_e1` 同样是 tuple，每项 shape `[natm, 3, nocc_σ, nocc_σ]`。
- 这里 PySCF 返回的是 `mo1_bra[σ]`，已经经过左乘 $C_{\mu p}^\sigma$（即 `solve_mo1` 内做了 bra-transform），是 $U_{\mu i}^{\mathbb{A}, \sigma}$ 而非原始的 $U_{p i}^{\mathbb{A}, \sigma}$。
  - **未** bra-transform 的 `mo1_a`/`mo1_b` 将在 `05-5` 中通过手写 `solve_withs1` 得到并存入 npz。

In [7]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")

    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1]
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [8]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
f1ao = [np.asarray(f1ao[x]) for x in (α, β)]  # each shape [natm, 3, nao, nao]
# 注意：PySCF 的 mf_hess.solve_mo1 返回的 mo1 是经过 bra-transform 的 C @ U，
# 即 U_{μi}^{A, σ}，而不是原始的 U_{pi}^{A, σ}。
# 这里命名为 mo1_bra 以与未 bra-transform 的 mo1（见 05-5）区分。
mo1_bra, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1_bra = [np.asarray(mo1_bra[x]) for x in (α, β)]  # each shape [natm, 3, nao, nocc_σ]
mo_e1 = [np.asarray(mo_e1[x]) for x in (α, β)]      # each shape [natm, 3, nocc_σ, nocc_σ]

print("f1ao shapes:    ", [f.shape for f in f1ao])
print("mo1_bra shapes :", [m.shape for m in mo1_bra])
print("mo_e1 shapes:   ", [m.shape for m in mo_e1])

f1ao shapes:     [(4, 3, 49, 49), (4, 3, 49, 49)]
mo1_bra shapes : [(4, 3, 49, 5), (4, 3, 49, 3)]
mo_e1 shapes:    [(4, 3, 5, 5), (4, 3, 3, 3)]


### CP-HF Hessian 公式核验

用 PySCF 跑出的 `f1ao`、`mo1_bra`、`mo_e1` 直接组装 CP-HF Hessian。RHF 中使用的因子 4（`4 * f1ao * dm1`）与 2（`2 * s1oo * mo_e1`）来自于：
- 闭壳层求和因子 2（U_{pi} 与 U_{pi}^* 各贡献一份）
- 密度矩阵的双占据因子 2

UHF 中，由于 `dm0[σ]` 不再带 `*2`、占据数恒为 1，因此对应的因子变为：
- `2 *` 替代 RHF 的 `4 *`（每个自旋通道单独贡献 2，再对 `σ ∈ {α, β}` 求和）
- `1 *` 替代 RHF 的 `2 *`

注意这里 `mo1_bra` 已是 bra-transform 后的 $U_{\mu i}^{\mathbb{A}, \sigma} = \sum_p C_{\mu p}^\sigma U_{p i}^{\mathbb{A}, \sigma}$。`05-5` 中我们会得到未 bra-transform 的 `mo1_a`/`mo1_b` 并存储到 npz。

In [9]:
de = np.zeros((natm, natm, 3, 3))

for i0, ia in enumerate(atmlst):
    shl0, shl1, p0, p1 = aoslices[ia]
    s1ao = ovlp_deriv1_generator(mol)(ia)
    s1oo = [mocc[x].T @ s1ao @ mocc[x] for x in (α, β)]

    for j0 in range(i0 + 1):
        ja = atmlst[j0]
        for x in (α, β):
            # f1ao term (mo1_bra 已是 C @ U)
            dm1 = mo1_bra[x][ja] @ mocc[x].T
            de[i0, j0] += 2 * (f1ao[x][ia][:, None] * dm1[None, :]).sum(axis=(-1, -2))
            # s1ao * energy term
            dm1 = mo1_bra[x][ja] @ (mocc[x] * eocc[x]).T
            de[i0, j0] -= 2 * (s1ao[:, None] * dm1[None, :]).sum(axis=(-1, -2))
            # s1oo * mo_e1 term
            de[i0, j0] -= 1 * (s1oo[x][:, None] * mo_e1[x][ja][None, :]).sum(axis=(-1, -2))

    for j0 in range(i0):
        de[j0, i0] = de[i0, j0].T

print("CP-HF Hessian assembled from PySCF mo1_bra/mo_e1:")
print("matches de_cphf:", np.allclose(de, de_cphf))

CP-HF Hessian assembled from PySCF mo1_bra/mo_e1:
matches de_cphf: True


In [10]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])

## f1ao 分解

skeleton Fock 一阶导数分为三部分：

$$
F^{\mathbb{A}, \sigma}_{\mu\nu} = H^{\mathbb{A}}_{\mu\nu} + J^{\mathbb{A}}_{\mu\nu}[D^\text{tot}] - K^{\mathbb{A}, \sigma}_{\mu\nu}[D^\sigma]
$$

其中：
- $H^{\mathbb{A}}$ 与自旋无关，与 RHF 形式完全相同。
- $J^{\mathbb{A}}$ 与自旋无关（依赖总密度 $D^\text{tot} = D^\alpha + D^\beta$）。**对比 RHF**：RHF 中 `dm0` 已含因子 2，UHF 中 `dm0_s = dm0_α + dm0_β` 无因子 2，因此 einsum 形式完全一致但缩并的对象不同。
- $K^{\mathbb{A}, \sigma}$ 按自旋通道分别构造，缩并对象是 $D^\sigma$（即 `mocc[σ] @ mocc[σ].T`）。

**与 RHF 的因子区别**：RHF 的 `k1ao` 在最终 `f1ao = h1ao + j1ao - 0.5 * k1ao` 中带 `0.5` 因子（来自闭壳层 K 公式 $-\tfrac{1}{2} D K$）。UHF 中每个自旋通道的 K 直接以 $-K^\sigma$ 形式出现，**无 `0.5` 因子**。

### Reference values

通过 `df.hessian.uhf._gen_jk` 获得参考值。PySCF 返回 `(h1, vj1, (vk1a, vk1b))`，其中 `h1` 与 `vj1` 仍是单一张量（与自旋无关），`vk1a/vk1b` 按自旋分通道。

In [11]:
mf_hess_aux0 = mf.Hessian()
mf_hess_aux0.auxbasis_response = 0
ref0 = list(df_uhf_hess._gen_jk(mf_hess_aux0, mo_coeff, mo_occ))
h1ao = np.array([r[1] for r in ref0])
j1ao_aux0 = np.array([r[2] for r in ref0])
# vk1 是 (vk1a, vk1b) 元组；按 [α, β, natm, 3, nao, nao] 整理
k1ao_aux0 = np.array([[r[3][x] for r in ref0] for x in (α, β)])
print("h1ao shape       :", h1ao.shape)
print("j1ao_aux0 shape  :", j1ao_aux0.shape)
print("k1ao_aux0 shape  :", k1ao_aux0.shape, "(spin, natm, 3, nao, nao)")

h1ao shape       : (4, 3, 49, 49)
j1ao_aux0 shape  : (4, 3, 49, 49)
k1ao_aux0 shape  : (2, 4, 3, 49, 49) (spin, natm, 3, nao, nao)


In [12]:
mf_hess_aux1 = mf.Hessian()
mf_hess_aux1.auxbasis_response = 1
ref1 = list(df_uhf_hess._gen_jk(mf_hess_aux1, mo_coeff, mo_occ))
j1ao_aux1 = np.array([r[2] for r in ref1]) - j1ao_aux0
k1ao_aux1 = np.array([[r[3][x] for r in ref1] for x in (α, β)]) - k1ao_aux0

In [13]:
# 核验：f1ao_σ = h1ao + j1ao - k1ao_σ （注意没有 RHF 中的 0.5 因子）
for x in (α, β):
    assert np.allclose(
        h1ao + j1ao_aux0 + j1ao_aux1 - (k1ao_aux0[x] + k1ao_aux1[x]),
        f1ao[x],
    )
print("f1ao decomposition verified for both spin channels.")

f1ao decomposition verified for both spin channels.


### h1ao part

与 RHF 完全相同，与自旋无关。直接复用 RHF 的 `hcore_deriv1_generator`。

In [14]:
def hcore_deriv1_generator(mol):
    h1 = - mol.intor("int1e_ipkin") - mol.intor("int1e_ipnuc")
    if mol.has_ecp():
        h1 -= mol.intor("ECPscalar_ipnuc")
    ecp_atoms = set(mol._ecpbas[:, gto.ATOM_OF])
    aoslices = mol.aoslice_by_atom()

    def get_hcore_deriv_at_atoms(A):
        _, _, p0, p1 = aoslices[A]
        z = mol.atom_charge(A)
        with mol.with_rinv_at_nucleus(A):
            h1ao = -z * mol.intor("int1e_iprinv")
            if A in ecp_atoms:
                h1ao += mol.intor("ECPscalar_iprinv")
        h1ao[:, p0:p1] += h1[:, p0:p1]
        return h1ao + h1ao.swapaxes(-1, -2)
    return get_hcore_deriv_at_atoms

In [15]:
for A in range(mol.natm):
    assert np.allclose(hcore_deriv1_generator(mol)(A), h1ao[A])
print("h1ao verified.")

h1ao verified.


### j1ao part

J 部分依赖总密度 `dm0_s`，einsum 形式与 RHF 完全相同。**注意**：RHF 中 `dm0` 已含因子 2，因此结果数值与 RHF 不会直接相同（这里 `dm0_s` 没有这个因子）。但与 PySCF 的 UHF 参考值一致。

In [16]:
%%time
scr1 = np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip1, int2c2e_inv, int3c2e, dm0_s)

j1ao_aux0_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    j1ao_aux0_recap[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00)
    j1ao_aux0_recap[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, kl -> tuv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0_s[slcA])
    j1ao_aux0_recap[A] -= 2 * scr2
assert np.allclose(j1ao_aux0_recap, j1ao_aux0)
print("j1ao_aux0 verified.")

j1ao_aux0 verified.
CPU times: user 32.8 ms, sys: 2.93 ms, total: 35.7 ms
Wall time: 4.6 ms


In [17]:
%%time
j1ao_aux1_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    j1ao_aux1_recap[A] -= np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, dm0_s)
    # (00|0)(1|00)
    j1ao_aux1_recap[A] -= np.einsum("uvP, PQ, tklQ, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], dm0_s)
    # (00|0)(1|0)(0|00)
    j1ao_aux1_recap[A] += np.einsum("uvP, PQ, tQR, RS, klS, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0_s)
    # (00|0)(0|1)(0|00)
    j1ao_aux1_recap[A] += np.einsum("uvP, PQ, tRQ, RS, klS, kl -> tuv", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, dm0_s)
assert np.allclose(j1ao_aux1_recap, j1ao_aux1)
print("j1ao_aux1 verified.")

j1ao_aux1 verified.
CPU times: user 82.8 ms, sys: 962 μs, total: 83.7 ms
Wall time: 10.5 ms


### k1ao part

K 部分按自旋通道分别构造。einsum 形式与 RHF 完全相同，只是把 `mocc_2`（RHF 下带 `sqrt(occ)` 缩放）换成 UHF 下对应的 `mocc_2[σ]`（UHF 中 occ=1，无缩放）。

In [18]:
k1ao_aux0_recap = np.zeros([2, natm, 3, nao, nao])
for x in (α, β):
    scr1 = np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x])
    for A in range(mol.natm):
        sh0, sh1, p0, p1 = aoslices[A]
        slcA = slice(p0, p1)
        # (10|0)(0|00)
        k1ao_aux0_recap[x, A, :, slcA, :] -= scr1[:, slcA, :]
        # (01|0)(0|00)
        k1ao_aux0_recap[x, A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
        # (00|0)(0|10), (00|0)(0|01)
        scr2 = np.einsum("tklP, PQ, uvQ, ki, ui -> tlv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2[x][slcA], mocc_2[x])
        k1ao_aux0_recap[x, A] -= scr2 + scr2.swapaxes(-1, -2)
assert np.allclose(k1ao_aux0_recap, k1ao_aux0)
print("k1ao_aux0 verified for both spin channels.")

k1ao_aux0 verified for both spin channels.


In [19]:
k1ao_aux1_recap = np.zeros([2, natm, 3, nao, nao])
for x in (α, β):
    for A in range(mol.natm):
        sh0, sh1, p0, p1 = auxslices[A]
        slcA = slice(p0, p1)
        # (00|1)(0|00)
        k1ao_aux1_recap[x, A] -= np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2[x], mocc_2[x])
        # (00|0)(1|00)
        k1ao_aux1_recap[x, A] -= np.einsum("uvP, PQ, tklQ, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], mocc_2[x], mocc_2[x])
        # (00|0)(1|0)(0|00)
        k1ao_aux1_recap[x, A] += np.einsum("uvP, PQ, tQR, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x])
        # (00|0)(0|1)(0|00)
        k1ao_aux1_recap[x, A] += np.einsum("uvP, PQ, tRQ, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2[x], mocc_2[x])
assert np.allclose(k1ao_aux1_recap, k1ao_aux1)
print("k1ao_aux1 verified for both spin channels.")

k1ao_aux1 verified for both spin channels.


## 存储到分解文件

把 PySCF 解出的 `mo1_bra`（bra-transformed）、`mo_e1` 也存到 npz，供后续 notebook 使用。由于两个自旋通道的形状不同，按 `_a`、`_b` 后缀分别保存。

未 bra-transform 的 `mo1_a`/`mo1_b` 将在 `05-5` 中通过手写 `solve_withs1` 得到并追加保存。

In [20]:
dat = dict(np.load("nh3_u_hf_decomp.npz"))
dat.update({
    "mo1_bra_a": mo1_bra[α],
    "mo1_bra_b": mo1_bra[β],
    "mo_e1_a": mo_e1[α],
    "mo_e1_b": mo_e1[β],
})
np.savez("nh3_u_hf_decomp.npz", **dat)